# Launcher: hip_fractures_segmentation runs on Kaggle

**One notebook for every run.** It clones the GitHub repository, then starts the run you name in `RUN`.
All code lives in the repository, so this notebook never needs editing beyond the settings cell.

**Before the first run (once):**
1. *Add Input* → attach the dataset **utopianstar/dataset**.
2. *Settings* → **Accelerator: GPU** (T4 x2 or P100) and **Internet: On** (needs a phone-verified account).

**Each run:**
1. Set `RUN` (and `BRANCH` if needed) in the next cell. The available runs are listed in `kaggle/runs.json`.
   Use `BRANCH = "main"` once the setup pull request is merged; before that, `"claude/project-thread-kges8j"`.
2. Click **Save Version** → **Save & Run All (Commit)**. The run continues in the background; you can close the tab.
3. When it finishes, open the version → *Output* → download `results_<RUN>.zip` into `results/_inbox/` in the local folder.

In [ ]:
# ==== Settings: change these for each run ====
RUN = "S00_env_check"    # name of the run in kaggle/runs.json (S = shared, A = Paper A, R = Paper B)
BRANCH = "main"          # branch or tag; before the setup pull request is merged use "claude/project-thread-kges8j"
EXTRA_ARGS = ""          # optional extra arguments for the run's script
REPO_URL = "https://github.com/nithoukhan1/hip_fractures_segmentation.git"

In [ ]:
# Clone the repository into /tmp/code (outside /kaggle/working, so the code is not packed into the output).
# Public repository: no token needed. If the repository is made private later, add a Kaggle secret named
# GITHUB_TOKEN (Add-ons -> Secrets) holding a read-only GitHub token; it is used automatically and never printed.
import os, shutil, subprocess
CODE = "/tmp/code"
shutil.rmtree(CODE, ignore_errors=True)
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", "https://x-access-token:" + UserSecretsClient().get_secret("GITHUB_TOKEN") + "@")
    print("Using the GITHUB_TOKEN secret")
except Exception:
    print("No GITHUB_TOKEN secret found: cloning the public repository")
r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, CODE], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError("git clone failed:\n" + r.stderr.replace(url, REPO_URL))
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--format=commit %H%n%s (%ci)"], capture_output=True, text=True).stdout)
if not os.path.exists(os.path.join(CODE, "kaggle", "run.py")):
    raise RuntimeError(f"Branch '{BRANCH}' does not contain the project code yet. Merge the setup pull request "
                       "on GitHub first, or set BRANCH = 'claude/project-thread-kges8j' in the settings cell.")

In [ ]:
# Start the run. Everything it writes goes to /kaggle/working/results/<RUN>/
!python /tmp/code/kaggle/run.py {RUN} --extra "{EXTRA_ARGS}"

In [ ]:
# Pack the run's results into one zip for easy download from the version's Output tab.
!cd /kaggle/working && zip -qr results_{RUN}.zip results/{RUN} && ls -lh /kaggle/working